# KRTX–KATX radar cross-section
Vertical cross-section along the straight (great-circle) line from KRTX (Portland) to KATX (Seattle).
The whole line lies on a single azimuth from each radar, so radial velocity *is* the along-line component
(sign flipped for KATX so that **positive = toward KATX**, i.e. roughly northward).

Two radar-combination methods:
- **closest**: use the nearer radar (falls back to the other where the nearer one has no beam, e.g. cone of silence)
- **blended**: same as closest, but in the middle third of the line average both radars (reflectivity averaged in linear Z)

Two products per method: reflectivity with along-line wind arrows, and shaded along-line velocity.

In [ ]:
import os
import warnings
import glob
from datetime import datetime, timedelta

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pyart
from pyproj import Geod

In [ ]:
# --- Configuration ---
data_dir   = './radar_files'
out_root   = './radar_animations/cross_section'
start_time = datetime(2024, 8, 18, 0, 0)
end_time   = datetime(2024, 8, 18, 6, 0)
frame_step = timedelta(minutes=5)
max_offset = timedelta(minutes=10)   # max allowed gap between frame time and a radar volume

ds_km      = 1.0                      # along-line grid spacing
z_km       = np.arange(0, 16.01, 0.25)  # altitude (MSL) grid
beamwidth  = 1.0                      # deg; NEXRAD half-power beamwidth
dealias    = True

radars = {
    'KRTX': dict(name='Portland', lat=45.71504, lon=-122.96500),
    'KATX': dict(name='Seattle',  lat=48.19461, lon=-122.49569),
}

# Geometry of the KRTX -> KATX line
geod = Geod(ellps='WGS84')
az_rtx, az_atx, line_m = geod.inv(radars['KRTX']['lon'], radars['KRTX']['lat'],
                                  radars['KATX']['lon'], radars['KATX']['lat'])
az_rtx, az_atx = az_rtx % 360, az_atx % 360
L = line_m / 1000
s_km = np.arange(0, L + ds_km / 2, ds_km)   # distance from KRTX
s_edges = np.concatenate([[s_km[0] - ds_km / 2], s_km + ds_km / 2])
z_edges = np.concatenate([[z_km[0] - 0.125], z_km + 0.125])

# Lat/lon along the line, for the top axis
_line = np.array(geod.fwd(np.full(s_km.size, radars['KRTX']['lon']), np.full(s_km.size, radars['KRTX']['lat']),
                          np.full(s_km.size, az_rtx), s_km * 1000)[:2])
line_lon, line_lat = _line
print(f'Line length {L:.1f} km; azimuth from KRTX {az_rtx:.2f} deg, from KATX {az_atx:.2f} deg')

In [ ]:
# --- Download missing volumes (unsigned access to the public NEXRAD bucket) ---
# Only contacts S3 if some 5-min frame in the window has no local volume within max_offset.
bucket = 'unidata-nexrad-level2'   # formerly noaa-nexrad-level2 (now access-denied)

def download_radar_files(station, t0, t1, output_dir=data_dir):
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    os.makedirs(output_dir, exist_ok=True)
    d = t0.date()
    while d <= t1.date():
        prefix = f'{d:%Y/%m/%d}/{station}/'
        for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get('Contents', []):
                basename = os.path.basename(obj['Key'])
                try:
                    t = datetime.strptime(basename[4:17], '%Y%m%d_%H%M')
                except ValueError:
                    continue
                local_path = os.path.join(output_dir, basename)
                if t0 <= t <= t1 and not os.path.exists(local_path):
                    print(f'Downloading {basename}')
                    s3.download_file(bucket, obj['Key'], local_path)
        d += timedelta(days=1)

def local_coverage_ok(station):
    times = [datetime.strptime(os.path.basename(f)[4:17], '%Y%m%d_%H%M')
             for f in glob.glob(os.path.join(data_dir, f'{station}*_V06'))]
    t = start_time
    while t <= end_time:
        if not any(abs(tt - t) <= max_offset for tt in times):
            return False
        t += frame_step
    return True

for code in radars:
    if local_coverage_ok(code):
        print(f'{code}: local files cover {start_time:%H%M}–{end_time:%H%M}Z, no download needed')
    else:
        download_radar_files(code, start_time - max_offset, end_time + max_offset)

In [ ]:
# --- Sample one radar volume along the line ---
def _pick_sweeps(radar, field):
    """One sweep per fixed angle that actually contains `field` (VCP 212 repeats 0.5 deg)."""
    picked = {}
    for i in range(radar.nsweeps):
        sl = radar.get_slice(i)
        if np.ma.count(radar.fields[field]['data'][sl]) == 0:
            continue
        ang = round(float(radar.fixed_angle['data'][i]), 1)
        picked.setdefault(ang, i)
    return sorted(picked.values())


def sample_along_line(radar, target_az, s_local):
    """Return {'reflectivity', 'velocity'} on (z_km, s_local) grids, NaN where no beam.

    s_local is ground distance (km) from this radar, evenly spaced. For each sweep, the 3 rays nearest
    target_az and all gates falling in each column are averaged (reflectivity in linear Z). Each grid
    cell then takes the sweep whose beam centre is closest in height, if the cell lies within that beam.
    Velocity is radial / cos(elevation), i.e. the horizontal component along the line, away from this radar.
    """
    alt_km = float(radar.altitude['data'][0]) / 1000
    r_km = radar.range['data'] / 1000
    step = s_local[1] - s_local[0]
    out = {}
    for field in ['reflectivity', 'velocity']:
        grid = np.full((z_km.size, s_local.size), np.nan)
        best = np.full(grid.shape, np.inf)
        for i in _pick_sweeps(radar, field):
            sl = radar.get_slice(i)
            az = radar.azimuth['data'][sl]
            daz = np.abs((az - target_az + 180) % 360 - 180)
            ray = int(np.argmin(daz))
            if daz[ray] > 1.0:
                continue
            rays = np.arange(ray - 1, ray + 2) % az.size
            elev = float(radar.fixed_angle['data'][i])
            vals = np.ma.filled(radar.fields[field]['data'][sl][rays].astype(float), np.nan)
            if field == 'reflectivity':
                vals = 10 ** (vals / 10)
            else:
                vals = vals / np.cos(np.radians(elev))

            x, y, zc = pyart.core.antenna_to_cartesian(r_km, target_az, elev)
            s_gate = np.hypot(x, y) / 1000
            zc = zc / 1000 + alt_km
            zlo = pyart.core.antenna_to_cartesian(r_km, target_az, elev - beamwidth / 2)[2] / 1000 + alt_km
            zhi = pyart.core.antenna_to_cartesian(r_km, target_az, elev + beamwidth / 2)[2] / 1000 + alt_km

            # Bin gates (from all 3 rays) into columns
            col = np.rint((s_gate - s_local[0]) / step).astype(int)
            col = np.broadcast_to(col, vals.shape)
            ok = np.isfinite(vals) & (col >= 0) & (col < s_local.size)
            n = np.bincount(col[ok], minlength=s_local.size)
            tot = np.bincount(col[ok], weights=vals[ok], minlength=s_local.size)
            with np.errstate(invalid='ignore', divide='ignore'):
                colval = tot / n
            if field == 'reflectivity':
                with np.errstate(divide='ignore'):
                    colval = 10 * np.log10(colval)

            in_range = (s_local >= s_gate[0]) & (s_local <= s_gate[-1])
            zc_c, zlo_c, zhi_c = (np.interp(s_local, s_gate, a) for a in (zc, zlo, zhi))
            dist = np.abs(z_km[:, None] - zc_c[None, :])
            inside = (z_km[:, None] >= zlo_c[None, :]) & (z_km[:, None] <= zhi_c[None, :]) & in_range[None, :]
            upd = inside & (dist < best)
            grid[upd] = np.broadcast_to(colval[None, :], grid.shape)[upd]
            best[upd] = dist[upd]
        out[field] = grid
    return out


def process_volume(fn, code):
    radar = pyart.io.read_nexrad_archive(fn)
    if dealias:
        vel_sweeps = _pick_sweeps(radar, 'velocity')
        radar = radar.extract_sweeps(sorted(set(vel_sweeps) | set(_pick_sweeps(radar, 'reflectivity'))))
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', UserWarning)
                dv = pyart.correct.dealias_region_based(radar)
            radar.add_field('velocity', dv, replace_existing=True)
        except Exception as e:
            print(f'  dealiasing failed for {os.path.basename(fn)}: {e}')
    if code == 'KRTX':
        return sample_along_line(radar, az_rtx, s_km)
    xs = sample_along_line(radar, az_atx, L - s_km)
    xs['velocity'] = -xs['velocity']   # away-from-KATX -> toward-KATX positive
    return xs

In [ ]:
# --- File lists and frame matching ---
def file_time(fn):
    return datetime.strptime(os.path.basename(fn)[4:17], '%Y%m%d_%H%M')

radar_files = {code: sorted(glob.glob(os.path.join(data_dir, f'{code}*_V06')), key=file_time) for code in radars}

def nearest_file(code, t):
    files = radar_files[code]
    if not files:
        return None
    fn = min(files, key=lambda f: abs(file_time(f) - t))
    return fn if abs(file_time(fn) - t) <= max_offset else None

frame_times = []
t = start_time
while t <= end_time:
    frame_times.append(t)
    t += frame_step

_cache = {}
def get_xsec(code, fn):
    if fn is None:
        return None
    if fn not in _cache:
        print(f'  reading {os.path.basename(fn)}')
        _cache[fn] = process_volume(fn, code)
    return _cache[fn]

In [ ]:
# --- Combine the two radars ---
def _dbz_mean(a, b):
    with np.errstate(invalid='ignore'), np.testing.suppress_warnings() as sup:
        sup.filter(RuntimeWarning)
        return 10 * np.log10(np.nanmean([10 ** (a / 10), 10 ** (b / 10)], axis=0))

def combine(rtx, atx, method):
    near_rtx = (s_km < L / 2)[None, :]
    middle = ((s_km >= L / 3) & (s_km <= 2 * L / 3))[None, :]
    out = {}
    for field in ['reflectivity', 'velocity']:
        a = rtx[field] if rtx is not None else np.full((z_km.size, s_km.size), np.nan)
        b = atx[field] if atx is not None else np.full((z_km.size, s_km.size), np.nan)
        prim, other = np.where(near_rtx, a, b), np.where(near_rtx, b, a)
        comb = np.where(np.isnan(prim), other, prim)
        if method == 'blended':
            if field == 'reflectivity':
                avg = _dbz_mean(a, b)
            else:
                with np.testing.suppress_warnings() as sup:
                    sup.filter(RuntimeWarning)
                    avg = np.nanmean([a, b], axis=0)
            comb = np.where(middle, avg, comb)
        out[field] = comb
    return out

In [ ]:
# --- Plotting ---
TITLES = {'reflectivity': 'Reflectivity + along-line wind', 'velocity': 'Along-line velocity'}
CB_LABELS = {'reflectivity': 'Reflectivity (dBZ)', 'velocity': 'Along-line velocity (m/s, + toward KATX)'}


def draw_xsec(ax, xs, product, method, xlabels=True, toplabels=True, quiverkey=True, fs=9):
    """Draw one cross-section onto `ax` and return the pcolormesh (for a colorbar)."""
    if product == 'reflectivity':
        pm = ax.pcolormesh(s_edges, z_edges, xs['reflectivity'], cmap='pyart_NWSRef', vmin=-20, vmax=75)
        # Along-line wind arrows, thinned
        js, ks = slice(4, None, 8), slice(1, None, 3)
        S, Z = np.meshgrid(s_km[js], z_km[ks])
        U = xs['velocity'][ks, js]
        ok = np.isfinite(U) & (xs['reflectivity'][ks, js] > 0)
        q = ax.quiver(S[ok], Z[ok], U[ok], np.zeros(ok.sum()), angles='xy', scale_units='x', scale=2.5,
                      width=0.0018, headwidth=4, headlength=4, color='k', zorder=3)
        if quiverkey:
            ax.quiverkey(q, 0.975, 0.94, 20, '20 m/s (+ toward KATX)', labelpos='W', coordinates='axes',
                         fontproperties={'size': fs - 1})
    else:
        pm = ax.pcolormesh(s_edges, z_edges, xs['velocity'], cmap='pyart_NWSVel', vmin=-30, vmax=30)

    if method == 'blended':
        for x in (L / 3, 2 * L / 3):
            ax.axvline(x, color='0.3', ls='--', lw=0.8)
    else:
        ax.axvline(L / 2, color='0.3', ls=':', lw=0.8)

    ax.set_xlim(0, L)
    ax.set_ylim(z_km[0], z_km[-1])
    if xlabels:
        ax.set_xlabel('Distance from KRTX along KRTX–KATX line (km)', fontsize=fs + 1)
    else:
        ax.tick_params(labelbottom=False)
    ax.set_ylabel('Altitude (km)', fontsize=fs + 1)
    ax.set_facecolor('0.92')
    ax.grid(color='0.6', lw=0.3, alpha=0.5)

    if toplabels:
        top = ax.secondary_xaxis('top', functions=(lambda s: np.interp(s, s_km, line_lat),
                                                   lambda lat: np.interp(lat, line_lat, s_km)))
        top.set_xticks(np.arange(46.0, 48.01, 0.5))
        top.set_xlabel('Latitude (°N)', fontsize=fs - 1)
        top.tick_params(labelsize=fs - 1)

    box = dict(facecolor='white', edgecolor='0.4', alpha=0.85, boxstyle='round,pad=0.25')
    ax.text(0.008, 0.025, 'KRTX (Portland)', transform=ax.transAxes, ha='left', va='bottom', fontsize=fs, bbox=box, zorder=5)
    ax.text(0.992, 0.025, 'KATX (Seattle)', transform=ax.transAxes, ha='right', va='bottom', fontsize=fs, bbox=box, zorder=5)
    return pm


def scan_str(tt):
    return f'{tt:%H:%M}' if tt is not None else 'n/a'


def plot_frame(xs, product, method, t, t_rtx, t_atx, out_png):
    fig, ax = plt.subplots(figsize=(11, 4.5), dpi=150)
    pm = draw_xsec(ax, xs, product, method)
    cb = fig.colorbar(pm, ax=ax, pad=0.01, fraction=0.04)
    cb.set_label(CB_LABELS[product], fontsize=9)
    ax.set_title(f'{TITLES[product]} ({method})   {t:%Y-%m-%d %H:%M} UTC\n'
                 f'KRTX scan {scan_str(t_rtx)}   KATX scan {scan_str(t_atx)}', fontsize=10, loc='left')
    fig.savefig(out_png, bbox_inches='tight')
    plt.close(fig)

In [ ]:
# --- Render all frames and GIFs ---
methods, products = ['closest', 'blended'], ['reflectivity', 'velocity']
frame_files = {(m, p): [] for m in methods for p in products}
for m in methods:
    for p in products:
        os.makedirs(os.path.join(out_root, m, p, 'frames'), exist_ok=True)

for i, t in enumerate(frame_times):
    print(t)
    f_rtx, f_atx = nearest_file('KRTX', t), nearest_file('KATX', t)
    rtx, atx = get_xsec('KRTX', f_rtx), get_xsec('KATX', f_atx)
    if rtx is None and atx is None:
        print('  no data, skipping')
        continue
    t_rtx = file_time(f_rtx) if f_rtx else None
    t_atx = file_time(f_atx) if f_atx else None
    for m in methods:
        xs = combine(rtx, atx, m)
        for p in products:
            out_png = os.path.join(out_root, m, p, 'frames', f'frame_{i:03d}.png')
            plot_frame(xs, p, m, t, t_rtx, t_atx, out_png)
            frame_files[(m, p)].append(out_png)

for (m, p), files in frame_files.items():
    gif_path = os.path.join(out_root, m, p, f'{p}_{start_time:%H%M}_{end_time:%H%M}.gif')
    imageio.mimsave(gif_path, [imageio.imread(f) for f in files], fps=4)
    print(f'Saved {len(files)} frames + GIF -> {gif_path}')

In [ ]:
# --- Multi-panel figure: reflectivity (left) and velocity (right) at selected times ---
panel_times  = [datetime(2024, 8, 18, h, 0) for h in (1, 2, 3, 4)]   # one row per time
panel_method = 'blended'
shared_cbar  = True    # one colorbar per column (False: one per panel, like the loop frames)
LETTERS = 'abcdefghijklmnopqrstuvwxyz'

def panel_label(ax, i, fontsize=9):
    ax.text(0.02, 0.98, f'({LETTERS[i % 26]})', transform=ax.transAxes, ha='left', va='top',
            fontsize=fontsize, fontweight='bold', zorder=20,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1.5))

products = ['reflectivity', 'velocity']
n = len(panel_times)
fig, axes = plt.subplots(n, 2, figsize=(15, 2.7 * n + 1.2), dpi=150, sharex=True, sharey=True,
                         layout='constrained', squeeze=False)
pms = {}
for r, t in enumerate(panel_times):
    f_rtx, f_atx = nearest_file('KRTX', t), nearest_file('KATX', t)
    xs = combine(get_xsec('KRTX', f_rtx), get_xsec('KATX', f_atx), panel_method)
    t_rtx = file_time(f_rtx) if f_rtx else None
    t_atx = file_time(f_atx) if f_atx else None
    for c, p in enumerate(products):
        ax = axes[r, c]
        pms[p] = draw_xsec(ax, xs, p, panel_method, xlabels=(r == n - 1), toplabels=(r == 0),
                           quiverkey=(r == 0), fs=8)
        panel_label(ax, 2 * r + c)
        head = f'{TITLES[p]}\n' if r == 0 else ''
        ax.set_title(f'{head}{t:%H:%M} UTC  (KRTX {scan_str(t_rtx)}, KATX {scan_str(t_atx)})',
                     fontsize=9, loc='left')
        if not shared_cbar:
            fig.colorbar(pms[p], ax=ax, pad=0.01, fraction=0.04).set_label(CB_LABELS[p], fontsize=8)
if shared_cbar:
    for c, p in enumerate(products):
        cb = fig.colorbar(pms[p], ax=axes[:, c], location='bottom', shrink=0.8, aspect=40, pad=0.01)
        cb.set_label(CB_LABELS[p], fontsize=9)

panel_png = os.path.join(out_root, panel_method,
                         f"xsec_panels_{'_'.join(f'{t:%H%M}' for t in panel_times)}.png")
fig.savefig(panel_png, bbox_inches='tight')
print('saved', panel_png)
plt.show()